# Step 6 — Ghép B2/H3-B/SUP-LR và phân tích bổ sung (WSL)

Notebook chạy phân tích offline trên các dự đoán đã có, không gọi LLM và không fit lại H3-B/anchor. Script kiểm tra lock, protocol hash, manifest, hash đầu vào, coverage ID và nhãn trước khi phân tích. B2 dùng run chính đã đăng ký, seed 42.

Các phân tích gồm P_pre/P_post trên cùng 3 fold group-stratified; repair/harm theo lớp, expert tham chiếu và routing 5%/10%/25% cho H3-B gate (phân tích chính) và SUP-LR (bổ sung). H5.3 tại q=10% báo paired cluster-bootstrap CI cho benefit−confidence (balanced accuracy, MCC), benefit−random (balanced accuracy), và số ca LLM sửa đúng so với cận 97,5% của expert ngẫu nhiên cùng tỷ lệ đoán 1. Bootstrap dùng 5.000 lượt và cluster unit đã khóa. Ngưỡng H3-B lấy từ train OOF; ngưỡng SUP-LR lấy từ validation. Test chỉ dùng đánh giá cuối, không fit mô hình/chọn ngưỡng.

Khi notebook chạy xong và các file `analysis_manifest.json` cùng hai `complementarity_step6.json` được kiểm tra, phần tính toán offline của Bước 6 hoàn tất. Chấm relevance, packet 60 ca, kiểm toán nhãn, audit prompt thủ công và thống kê/bảng cuối vẫn là hạng mục riêng nếu chưa làm.


In [1]:
from pathlib import Path
import json, subprocess, sys

SUMMARY_ROOT = Path('/mnt/d/JUNIOR/LabThayTho/GRACE-summary')
STEP6_ROOT = SUMMARY_ROOT / 'STEP6_supervised_anchors'
SCRIPT = STEP6_ROOT / 'src' / 'step6_complementarity.py'
B2_ROOT = (SUMMARY_ROOT / '01_Author_Baseline_Unified' / 'kaggle_qwen_q4_b2' /
           'downloaded_output_20261101_run3' / 'grace_b2_four_split_8h')

for required in (STEP6_ROOT / 'step6_lock.json',
                 STEP6_ROOT / 'cache' / 'anchor_npz' / 'adapter_manifest.json',
                 STEP6_ROOT / 'runs' / 'h3b_locked', SCRIPT,
                 SUMMARY_ROOT / 'configs' / 'protocol.yaml', B2_ROOT):
    assert required.exists(), f'Missing required input: {required}'

print('Python:', sys.executable)
print('Step 6 root:', STEP6_ROOT)
print('B2 source:', B2_ROOT)
print('Analysis:', SCRIPT)

Python: /mnt/d/JUNIOR/LabThayTho/GRACE-summary/.venv-wsl/bin/python
Step 6 root: /mnt/d/JUNIOR/LabThayTho/GRACE-summary/STEP6_supervised_anchors
B2 source: /mnt/d/JUNIOR/LabThayTho/GRACE-summary/01_Author_Baseline_Unified/kaggle_qwen_q4_b2/downloaded_output_20261101_run3/grace_b2_four_split_8h
Analysis: /mnt/d/JUNIOR/LabThayTho/GRACE-summary/STEP6_supervised_anchors/src/step6_complementarity.py


## Chạy ghép và phân tích

Cell kế tiếp chạy paired bootstrap 5.000 lượt và routing trên hai dataset, nên có thể mất vài phút. Nếu kiểm tra provenance/ID/label thất bại, dừng và xử lý lỗi; không sửa lock để ép chạy qua.

In [2]:
command = [sys.executable, str(SCRIPT),
           '--step6-root', str(STEP6_ROOT),
           '--summary-root', str(SUMMARY_ROOT),
           '--b2-root', str(B2_ROOT)]
print('Running:', ' '.join(command), flush=True)
subprocess.run(command, check=True, cwd=SUMMARY_ROOT)

Running: /mnt/d/JUNIOR/LabThayTho/GRACE-summary/.venv-wsl/bin/python /mnt/d/JUNIOR/LabThayTho/GRACE-summary/STEP6_supervised_anchors/src/step6_complementarity.py --step6-root /mnt/d/JUNIOR/LabThayTho/GRACE-summary/STEP6_supervised_anchors --summary-root /mnt/d/JUNIOR/LabThayTho/GRACE-summary --b2-root /mnt/d/JUNIOR/LabThayTho/GRACE-summary/01_Author_Baseline_Unified/kaggle_qwen_q4_b2/downloaded_output_20261101_run3/grace_b2_four_split_8h
Join PASS: devign: validation=2732, test=2732
Join PASS: reveal: validation=2273, test=2274
Analysis complete: devign
Analysis complete: reveal
Wrote Step 6 analysis to /mnt/d/JUNIOR/LabThayTho/GRACE-summary/STEP6_supervised_anchors/runs/complementarity_step6/step6_lock_64fb2c99d904_b5d441d3


CompletedProcess(args=['/mnt/d/JUNIOR/LabThayTho/GRACE-summary/.venv-wsl/bin/python', '/mnt/d/JUNIOR/LabThayTho/GRACE-summary/STEP6_supervised_anchors/src/step6_complementarity.py', '--step6-root', '/mnt/d/JUNIOR/LabThayTho/GRACE-summary/STEP6_supervised_anchors', '--summary-root', '/mnt/d/JUNIOR/LabThayTho/GRACE-summary', '--b2-root', '/mnt/d/JUNIOR/LabThayTho/GRACE-summary/01_Author_Baseline_Unified/kaggle_qwen_q4_b2/downloaded_output_20261101_run3/grace_b2_four_split_8h'], returncode=0)

## Xác nhận artifact cuối

Đọc manifest của run vừa tạo, xác nhận có kết quả cho Devign/ReVeal và lưu đường dẫn để mở JSON. Không kết luận nghiên cứu từ một mình descriptive metrics; diễn giải CI/verdict cùng các kiểm tra thủ công ở checklist nghiên cứu.

In [3]:
analysis_root = STEP6_ROOT / 'runs' / 'complementarity_step6'
manifests = sorted(analysis_root.glob('step6_lock_*_*/analysis_manifest.json'),
                   key=lambda p: p.stat().st_mtime, reverse=True)
assert manifests, f'No analysis manifest found under {analysis_root}'
manifest_path = manifests[0]
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
run_dir = manifest_path.parent
for dataset in ('devign', 'reveal'):
    result_path = run_dir / dataset / 'complementarity_step6.json'
    assert result_path.is_file(), f'Missing result: {result_path}'
    result = json.loads(result_path.read_text(encoding='utf-8'))
    assert result['n_validation'] > 0 and result['n_test'] > 0
    h5_3 = result['h3b_gate_vs_b2']['h5_3_registered_checks']
    assert h5_3['budget'] == 0.10 and 'criteria' in h5_3
    print(dataset, 'validation=', result['n_validation'],
          'test=', result['n_test'],
          'B2 valid test=', result['b2_valid_n_test'])
    print('  H5.3 routing checks:', h5_3['criteria'],
          'all met=', h5_3['all_routing_criteria_met'])
print('Run directory:', run_dir)
print('Manifest:', manifest_path)
print('Input joins, result hashes, protocol/lock hashes recorded:',
      bool(manifest.get('inputs')) and bool(manifest.get('output_hashes')))


devign validation= 2732 test= 2732 B2 valid test= 2732
  H5.3 routing checks: {'benefit_vs_confidence_ba_upper_ci_lt_0_02': True, 'benefit_vs_confidence_mcc_upper_ci_lt_0_02': False, 'benefit_vs_random_ba_upper_ci_lt_0_01': False, 'llm_repairs_not_above_random_expert_97_5_percentile': True} all met= False
reveal validation= 2273 test= 2274 B2 valid test= 2274
  H5.3 routing checks: {'benefit_vs_confidence_ba_upper_ci_lt_0_02': True, 'benefit_vs_confidence_mcc_upper_ci_lt_0_02': True, 'benefit_vs_random_ba_upper_ci_lt_0_01': True, 'llm_repairs_not_above_random_expert_97_5_percentile': True} all met= True
Run directory: /mnt/d/JUNIOR/LabThayTho/GRACE-summary/STEP6_supervised_anchors/runs/complementarity_step6/step6_lock_64fb2c99d904_b5d441d3
Manifest: /mnt/d/JUNIOR/LabThayTho/GRACE-summary/STEP6_supervised_anchors/runs/complementarity_step6/step6_lock_64fb2c99d904_b5d441d3/analysis_manifest.json
Input joins, result hashes, protocol/lock hashes recorded: True
